<a href="https://colab.research.google.com/github/deruke/aisecops-labs/blob/main/notebooks/Lab9_Security_Agent_Tool_Calling.ipynb" target="_new"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 9: Building a Security Agent with Tool Calling
## Teaching an LLM to Use Security Tools Autonomously

**Workshop:** Attacking, Defending & Leveraging Agentic AI  
**Authors:** Derek Banks and Dr. Brian Fehrman  
**Time:** 60 minutes  
**Platform:** Google Colab (T4 recommended)  
**Target Model:** Qwen2.5-3B-Instruct (4-bit quantized)  

---

### What You Will Learn
- How to build an AI agent that uses **tool calling** to perform security tasks
- Implementing security-relevant tools: IP reputation, CVE lookup, log analysis, and more
- How the agent **reasons** about which tool to call and **chains** results together
- The iterative agentic loop: **Plan -> Act -> Observe -> Reflect**
- The real risks of **excessive agency** in security automation

### Prerequisites
- Basic understanding of LLMs and prompt engineering
- Familiarity with Python
- Completion of Labs 1-8 (or equivalent background)
- A Google account for Colab access

### Where This Fits
This is the **Leveraging AI** lab. In previous labs, you attacked models (prompt injection,  
jailbreaking, abliteration) and defended them (guardrails, alignment). Now you will **build  
something useful** -- a security agent that demonstrates how agentic AI actually works  
under the hood.

**Reference:** Slides 207-220 (Agentic AI Architecture)

### Key References
- ReAct: Yao et al. (2023) "ReAct: Synergizing Reasoning and Acting in Language Models"
- OWASP LLM Top 10: LLM08 - Excessive Agency
- Anthropic (2024): "Building effective agents"

---
## Cell 1: Environment Setup
---

In [ ]:
# ============================================================
# Cell 1: Environment Setup
# ============================================================
# Install required packages and validate GPU availability.
# We load Qwen2.5-3B-Instruct in 4-bit quantization so it
# fits comfortably in T4 VRAM (~4 GB footprint).
# ============================================================

!pip install -q transformers accelerate bitsandbytes torch requests

import torch
import sys
import json
import re
import time
from datetime import datetime, timedelta
import random
import hashlib

print("=" * 60)
print("ENVIRONMENT CHECK")
print("=" * 60)
print(f"Python version : {sys.version.split()[0]}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"GPU            : {gpu_name}")
    print(f"GPU Memory     : {gpu_mem:.1f} GB")
else:
    print("WARNING: No GPU detected. This lab will run slowly on CPU.")
    print("         Go to Runtime -> Change runtime type -> T4 GPU")

print("=" * 60)
print("Environment ready.")

In [ ]:
# ============================================================
# Cell 2: Load the Language Model
# ============================================================
# Load Qwen2.5-3B-Instruct with 4-bit quantization.
# This is the "brain" of our security agent.
# Falls back to 1.5B if memory is constrained.
# ============================================================

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Try 3B first, fall back to 1.5B if memory constrained
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

try:
    print(f"Loading {MODEL_ID} in 4-bit...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
    print(f"Model loaded successfully: {MODEL_ID}")
except Exception as e:
    print(f"Failed to load 3B model: {e}")
    MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
    print(f"Falling back to {MODEL_ID}...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
    print(f"Fallback model loaded: {MODEL_ID}")

# Verify
param_count = sum(p.numel() for p in model.parameters())
print(f"Parameters: {param_count / 1e9:.2f}B")
if torch.cuda.is_available():
    mem_used = torch.cuda.memory_allocated() / 1e9
    print(f"GPU memory used: {mem_used:.2f} GB")
print("Agent brain loaded and ready.")

---
## Background: What Is an AI Agent?
---

An **AI agent** is an LLM that can take **actions** in the world, not just generate text.  
The key difference between a chatbot and an agent:

| | Chatbot | Agent |
|---|---------|-------|
| Input | User message | User message + tool results |
| Output | Text reply | Text reply OR tool call |
| State | Stateless or simple history | Maintains investigation state |
| Loop | Single turn | Multi-turn with tool execution |

### The Agentic Loop (Slide 213)

```
                    +------------------+
                    |   User Query     |
                    +--------+---------+
                             |
                             v
                    +------------------+
               +--->|    OBSERVE       |  <-- Receive input or tool results
               |    +--------+---------+
               |             |
               |             v
               |    +------------------+
               |    |     THINK        |  <-- Reason about what to do next
               |    +--------+---------+
               |             |
               |             v
               |    +------------------+
               |    |      ACT         |  <-- Call a tool OR give final answer
               |    +--------+---------+
               |             |
               |       Tool call?       
               |      /          \      
               |    YES           NO    
               |    /               \   
               |   v                 v  
               |  Execute         +------------------+
               |  Tool            |  FINAL ANSWER    |
               |   |              +------------------+
               +---+
```

### How Tool Calling Works

1. We define **tools** -- Python functions that do specific things (look up IPs, search CVEs, etc.)
2. We tell the LLM about these tools in its **system prompt** (name, purpose, parameters)
3. When the LLM needs information, it outputs a **structured tool call** instead of a text answer
4. Our code **parses** the tool call, **executes** the function, and feeds the result back
5. The LLM **continues reasoning** with the new information

This is exactly how ChatGPT plugins, Claude tool use, and frameworks like LangChain work  
under the hood. Today you build it from scratch.

### Why This Matters for Security

Security analysts spend hours on repetitive investigation tasks:
- Looking up suspicious IPs across multiple threat intel feeds
- Cross-referencing CVEs with asset inventories
- Triaging alerts by correlating log data

An AI agent can **automate the boring parts** while keeping a human in the loop  
for decisions that matter. But it also introduces new risks -- which we will explore.

---
## Part 1: Building Security Tools
---

We will build six security tools. Each returns **structured JSON data** that the  
agent can parse and reason about. In production, these would call real APIs  
(VirusTotal, Shodan, NIST NVD, etc.). For this lab, they return **realistic  
simulated data** so we can work without API keys.

**The tools:**
1. `ip_reputation` -- IP geolocation, ASN, reputation, malware associations
2. `cve_lookup` -- CVE details, CVSS score, affected products, exploit status
3. `log_analyzer` -- Parse raw logs, identify anomalies
4. `port_scanner` -- Simulated port scan with service detection
5. `whois_lookup` -- Domain registration data
6. `threat_intel` -- IOC enrichment (IP, hash, domain)

In [ ]:
# ============================================================
# Cell 3: Security Tool -- ip_reputation
# ============================================================
# Simulates an IP reputation lookup (like VirusTotal, AbuseIPDB).
# Returns geolocation, ASN, reputation score, and associations.
# ============================================================

def ip_reputation(ip: str) -> dict:
    """
    Look up the reputation and geolocation data for an IP address.
    Returns risk score, geolocation, ASN, and known associations.
    """
    # Deterministic seed from IP so results are consistent
    seed = int(hashlib.md5(ip.encode()).hexdigest()[:8], 16)
    rng = random.Random(seed)

    # Known "bad" IPs for demo scenarios
    known_bad = {
        "45.33.32.156": {
            "ip": "45.33.32.156",
            "country": "United States",
            "city": "Fremont, CA",
            "asn": "AS63949",
            "org": "Linode LLC",
            "risk_score": 72,
            "risk_level": "HIGH",
            "categories": ["scanning", "brute-force"],
            "last_seen_malicious": "2026-02-28",
            "malware_families": ["Mirai", "Hajime"],
            "abuse_reports": 147,
            "first_seen": "2024-03-15",
            "blocklists": ["Spamhaus DROP", "Emerging Threats"],
            "confidence": 0.89
        },
        "198.51.100.23": {
            "ip": "198.51.100.23",
            "country": "Russia",
            "city": "Moscow",
            "asn": "AS48666",
            "org": "Marosnet Telecom",
            "risk_score": 95,
            "risk_level": "CRITICAL",
            "categories": ["C2 server", "malware distribution", "phishing"],
            "last_seen_malicious": "2026-03-01",
            "malware_families": ["Cobalt Strike", "SystemBC"],
            "abuse_reports": 523,
            "first_seen": "2023-08-10",
            "blocklists": ["Spamhaus DROP", "Emerging Threats", "AlienVault OTX", "Feodo Tracker"],
            "confidence": 0.96
        },
        "203.0.113.50": {
            "ip": "203.0.113.50",
            "country": "China",
            "city": "Beijing",
            "asn": "AS4808",
            "org": "China Unicom Beijing",
            "risk_score": 88,
            "risk_level": "HIGH",
            "categories": ["APT activity", "data exfiltration"],
            "last_seen_malicious": "2026-02-25",
            "malware_families": ["PlugX", "ShadowPad"],
            "abuse_reports": 89,
            "first_seen": "2024-01-20",
            "blocklists": ["Emerging Threats", "AlienVault OTX"],
            "confidence": 0.92
        }
    }

    if ip in known_bad:
        return known_bad[ip]

    # Generate plausible data for unknown IPs
    countries = ["United States", "Germany", "Netherlands", "United Kingdom",
                 "Singapore", "Japan", "Brazil", "India", "Canada", "France"]
    risk_score = rng.randint(5, 65)

    return {
        "ip": ip,
        "country": rng.choice(countries),
        "city": "Unknown",
        "asn": f"AS{rng.randint(1000, 99999)}",
        "org": rng.choice(["Amazon AWS", "DigitalOcean", "Google Cloud",
                           "OVH SAS", "Hetzner", "Cloudflare"]),
        "risk_score": risk_score,
        "risk_level": "LOW" if risk_score < 30 else "MEDIUM" if risk_score < 60 else "HIGH",
        "categories": [] if risk_score < 30 else [rng.choice(["scanning", "spam", "proxy"])],
        "last_seen_malicious": "N/A" if risk_score < 30 else "2026-02-15",
        "malware_families": [],
        "abuse_reports": rng.randint(0, 20) if risk_score < 50 else rng.randint(20, 100),
        "first_seen": "2025-06-01",
        "blocklists": [],
        "confidence": round(rng.uniform(0.6, 0.85), 2)
    }

# Test it
print("=== Tool Test: ip_reputation ===")
result = ip_reputation("45.33.32.156")
print(json.dumps(result, indent=2))

In [ ]:
# ============================================================
# Cell 4: Security Tool -- cve_lookup
# ============================================================
# Simulates a CVE database lookup (like NIST NVD, MITRE).
# Returns CVE details, CVSS score, affected products, and
# exploit availability.
# ============================================================

def cve_lookup(cve_id: str) -> dict:
    """
    Look up a CVE by its ID. Returns description, CVSS score,
    affected products, and exploit availability.
    """
    known_cves = {
        "CVE-2024-3400": {
            "cve_id": "CVE-2024-3400",
            "description": "A command injection vulnerability in the GlobalProtect feature "
                           "of Palo Alto Networks PAN-OS software. An unauthenticated attacker "
                           "can execute arbitrary code with root privileges on the firewall.",
            "cvss_score": 10.0,
            "cvss_severity": "CRITICAL",
            "cvss_vector": "CVSS:3.1/AV:N/AC:L/PR:N/UI:N/S:C/C:H/I:H/A:H",
            "affected_products": [
                "PAN-OS 11.1 < 11.1.2-h3",
                "PAN-OS 11.0 < 11.0.4-h1",
                "PAN-OS 10.2 < 10.2.9-h1"
            ],
            "vendor": "Palo Alto Networks",
            "published_date": "2024-04-12",
            "last_modified": "2024-04-20",
            "exploit_available": True,
            "exploit_sources": ["Metasploit", "GitHub PoC", "Exploit-DB"],
            "actively_exploited": True,
            "cisa_kev": True,
            "patch_available": True,
            "patch_url": "https://security.paloaltonetworks.com/CVE-2024-3400",
            "references": [
                "https://nvd.nist.gov/vuln/detail/CVE-2024-3400",
                "https://unit42.paloaltonetworks.com/cve-2024-3400/"
            ],
            "confidence": 0.99
        },
        "CVE-2024-21887": {
            "cve_id": "CVE-2024-21887",
            "description": "A command injection vulnerability in web components of Ivanti "
                           "Connect Secure and Ivanti Policy Secure allows an authenticated "
                           "administrator to send specially crafted requests and execute "
                           "arbitrary commands on the appliance.",
            "cvss_score": 9.1,
            "cvss_severity": "CRITICAL",
            "cvss_vector": "CVSS:3.1/AV:N/AC:L/PR:H/UI:N/S:C/C:H/I:H/A:H",
            "affected_products": [
                "Ivanti Connect Secure 9.x",
                "Ivanti Connect Secure 22.x",
                "Ivanti Policy Secure"
            ],
            "vendor": "Ivanti",
            "published_date": "2024-01-10",
            "last_modified": "2024-02-15",
            "exploit_available": True,
            "exploit_sources": ["Metasploit", "GitHub PoC"],
            "actively_exploited": True,
            "cisa_kev": True,
            "patch_available": True,
            "patch_url": "https://forums.ivanti.com/s/article/CVE-2024-21887",
            "references": [
                "https://nvd.nist.gov/vuln/detail/CVE-2024-21887"
            ],
            "confidence": 0.98
        },
        "CVE-2023-44228": {
            "cve_id": "CVE-2023-44228",
            "description": "Apache Log4j2 JNDI features do not protect against attacker "
                           "controlled LDAP and other JNDI related endpoints. An attacker "
                           "who can control log messages or log message parameters can "
                           "execute arbitrary code.",
            "cvss_score": 10.0,
            "cvss_severity": "CRITICAL",
            "cvss_vector": "CVSS:3.1/AV:N/AC:L/PR:N/UI:N/S:C/C:H/I:H/A:H",
            "affected_products": [
                "Apache Log4j 2.0-beta9 through 2.15.0"
            ],
            "vendor": "Apache Software Foundation",
            "published_date": "2021-12-10",
            "last_modified": "2023-11-06",
            "exploit_available": True,
            "exploit_sources": ["Metasploit", "GitHub PoC", "Exploit-DB", "Nuclei"],
            "actively_exploited": True,
            "cisa_kev": True,
            "patch_available": True,
            "patch_url": "https://logging.apache.org/log4j/2.x/security.html",
            "references": [
                "https://nvd.nist.gov/vuln/detail/CVE-2021-44228"
            ],
            "confidence": 0.99
        }
    }

    if cve_id in known_cves:
        return known_cves[cve_id]

    # For unknown CVEs, return a not-found response
    return {
        "cve_id": cve_id,
        "description": "CVE not found in database.",
        "cvss_score": None,
        "cvss_severity": "UNKNOWN",
        "affected_products": [],
        "exploit_available": False,
        "actively_exploited": False,
        "patch_available": False,
        "confidence": 0.0
    }

# Test it
print("=== Tool Test: cve_lookup ===")
result = cve_lookup("CVE-2024-3400")
print(json.dumps(result, indent=2))

In [ ]:
# ============================================================
# Cell 5: Security Tool -- log_analyzer
# ============================================================
# Takes raw log entries (as a string) and identifies anomalies,
# suspicious patterns, and potential threats.
# ============================================================

def log_analyzer(log_entries: str) -> dict:
    """
    Analyze raw log entries for security anomalies.
    Accepts a string of newline-separated log entries.
    Returns structured findings with severity levels.
    """
    lines = [l.strip() for l in log_entries.strip().split("\n") if l.strip()]
    findings = []
    suspicious_ips = set()
    stats = {
        "total_lines": len(lines),
        "failed_logins": 0,
        "successful_logins": 0,
        "error_count": 0,
        "unique_ips": set(),
        "suspicious_paths": 0,
    }

    # Pattern-based detection rules
    suspicious_patterns = [
        (r"(?i)failed\s+(password|login|auth)", "BRUTE_FORCE", "HIGH",
         "Failed authentication attempt detected"),
        (r"(?i)(\.\.[\\/]|path\s*traversal)", "PATH_TRAVERSAL", "CRITICAL",
         "Directory traversal attempt detected"),
        (r"(?i)(union\s+select|;\s*drop\s+|'\s*or\s+'1'\s*=\s*'1)", "SQL_INJECTION", "CRITICAL",
         "SQL injection attempt detected"),
        (r"(?i)(<script|javascript:|on\w+=)", "XSS", "HIGH",
         "Cross-site scripting attempt detected"),
        (r"(?i)(cmd\.exe|/bin/(sh|bash)|powershell)", "COMMAND_INJECTION", "CRITICAL",
         "Command injection attempt detected"),
        (r"(?i)(\b4(0[0-9]|1[0-9]|2[0-9]|3[0-9])\b)", "HTTP_ERROR", "LOW",
         "HTTP client error detected"),
        (r"(?i)(\b5\d{2}\b)", "SERVER_ERROR", "MEDIUM",
         "HTTP server error detected"),
        (r"(?i)(root|admin|administrator).*(?:login|session|auth)", "PRIV_ESCALATION", "HIGH",
         "Privileged account activity detected"),
        (r"(?i)(wget|curl|nc\s+-e|ncat).*(?:http|ftp)", "DATA_EXFIL", "HIGH",
         "Potential data exfiltration tool usage"),
        (r"(?i)(nmap|masscan|zmap|nikto|sqlmap)", "RECON", "MEDIUM",
         "Reconnaissance tool signature detected"),
    ]

    # Extract IPs
    ip_pattern = re.compile(r"\b(?:\d{1,3}\.){3}\d{1,3}\b")

    for i, line in enumerate(lines):
        # Extract IPs from this line
        found_ips = ip_pattern.findall(line)
        stats["unique_ips"].update(found_ips)

        # Count auth events
        if re.search(r"(?i)failed\s+(password|login|auth)", line):
            stats["failed_logins"] += 1
        if re.search(r"(?i)(accepted|successful)\s+(password|login|auth)", line):
            stats["successful_logins"] += 1

        # Check suspicious patterns
        for pattern, category, severity, description in suspicious_patterns:
            if re.search(pattern, line):
                finding = {
                    "line_number": i + 1,
                    "category": category,
                    "severity": severity,
                    "description": description,
                    "source_ips": found_ips,
                    "log_excerpt": line[:200],
                }
                findings.append(finding)
                suspicious_ips.update(found_ips)
                break  # One finding per line (highest priority)

    # Detect brute force (many failed logins from same IP)
    if stats["failed_logins"] >= 5:
        findings.append({
            "line_number": "aggregate",
            "category": "BRUTE_FORCE_CAMPAIGN",
            "severity": "CRITICAL",
            "description": f"Brute force campaign: {stats['failed_logins']} failed login attempts detected",
            "source_ips": list(suspicious_ips),
            "log_excerpt": "Multiple lines -- aggregate detection",
        })

    # Sort findings by severity
    severity_order = {"CRITICAL": 0, "HIGH": 1, "MEDIUM": 2, "LOW": 3}
    findings.sort(key=lambda f: severity_order.get(f["severity"], 4))

    return {
        "analysis_timestamp": datetime.now().isoformat(),
        "total_lines_analyzed": stats["total_lines"],
        "unique_ips_found": list(stats["unique_ips"]),
        "suspicious_ips": list(suspicious_ips),
        "summary": {
            "total_findings": len(findings),
            "critical": sum(1 for f in findings if f["severity"] == "CRITICAL"),
            "high": sum(1 for f in findings if f["severity"] == "HIGH"),
            "medium": sum(1 for f in findings if f["severity"] == "MEDIUM"),
            "low": sum(1 for f in findings if f["severity"] == "LOW"),
        },
        "findings": findings[:15],  # Limit to top 15
        "confidence": 0.85
    }

# Test it
test_logs = """2026-03-01 10:15:23 sshd[1234]: Failed password for root from 198.51.100.23 port 22
2026-03-01 10:15:24 sshd[1235]: Failed password for admin from 198.51.100.23 port 22
2026-03-01 10:15:25 sshd[1236]: Failed password for root from 198.51.100.23 port 22
2026-03-01 10:15:30 httpd[5678]: 203.0.113.50 GET /admin/../../../etc/passwd 404
2026-03-01 10:15:35 httpd[5679]: 203.0.113.50 GET /search?q=1'+OR+'1'='1 200"""

print("=== Tool Test: log_analyzer ===")
result = log_analyzer(test_logs)
print(json.dumps(result, indent=2, default=str))

In [ ]:
# ============================================================
# Cell 6: Security Tool -- port_scanner
# ============================================================
# Simulates an nmap-style port scan with service detection.
# ============================================================

def port_scanner(target: str) -> dict:
    """
    Perform a simulated port scan on the target host.
    Returns open ports with service identification and versions.
    """
    seed = int(hashlib.md5(target.encode()).hexdigest()[:8], 16)
    rng = random.Random(seed)

    # Pre-defined scan results for demo targets
    known_targets = {
        "45.33.32.156": {
            "target": "45.33.32.156",
            "scan_time": "2.34s",
            "host_status": "up",
            "open_ports": [
                {"port": 22, "protocol": "tcp", "state": "open", "service": "ssh",
                 "version": "OpenSSH 8.9p1", "banner": "SSH-2.0-OpenSSH_8.9p1"},
                {"port": 80, "protocol": "tcp", "state": "open", "service": "http",
                 "version": "nginx 1.24.0", "banner": ""},
                {"port": 443, "protocol": "tcp", "state": "open", "service": "https",
                 "version": "nginx 1.24.0", "banner": ""},
                {"port": 9090, "protocol": "tcp", "state": "open", "service": "http-proxy",
                 "version": "unknown", "banner": ""},
            ],
            "os_guess": "Linux 5.x (Ubuntu)",
            "filtered_ports": 996,
            "confidence": 0.90
        },
        "suspicious-domain.xyz": {
            "target": "suspicious-domain.xyz",
            "resolved_ip": "198.51.100.23",
            "scan_time": "3.78s",
            "host_status": "up",
            "open_ports": [
                {"port": 80, "protocol": "tcp", "state": "open", "service": "http",
                 "version": "Apache 2.4.41", "banner": ""},
                {"port": 443, "protocol": "tcp", "state": "open", "service": "https",
                 "version": "Apache 2.4.41", "banner": ""},
                {"port": 4444, "protocol": "tcp", "state": "open", "service": "unknown",
                 "version": "unknown", "banner": "\x00\x01\x00"},
                {"port": 8443, "protocol": "tcp", "state": "open", "service": "https-alt",
                 "version": "Cobalt Strike Beacon", "banner": ""},
            ],
            "os_guess": "Linux 4.x (Debian)",
            "filtered_ports": 996,
            "confidence": 0.88
        }
    }

    if target in known_targets:
        return known_targets[target]

    # Generate plausible data for unknown targets
    common_ports = [
        (22, "ssh", "OpenSSH 8.2p1"), (80, "http", "nginx 1.18"),
        (443, "https", "nginx 1.18"), (3306, "mysql", "MySQL 8.0"),
        (5432, "postgresql", "PostgreSQL 14.0"), (8080, "http-proxy", "Apache Tomcat 9.0"),
        (6379, "redis", "Redis 7.0"), (27017, "mongodb", "MongoDB 6.0"),
    ]
    selected = rng.sample(common_ports, k=rng.randint(2, 5))

    return {
        "target": target,
        "scan_time": f"{rng.uniform(1.0, 5.0):.2f}s",
        "host_status": "up",
        "open_ports": [
            {"port": p, "protocol": "tcp", "state": "open",
             "service": s, "version": v, "banner": ""}
            for p, s, v in sorted(selected)
        ],
        "os_guess": rng.choice(["Linux 5.x", "Windows Server 2022", "FreeBSD 13"]),
        "filtered_ports": 1000 - len(selected),
        "confidence": round(rng.uniform(0.70, 0.90), 2)
    }

# Test it
print("=== Tool Test: port_scanner ===")
result = port_scanner("suspicious-domain.xyz")
print(json.dumps(result, indent=2))

In [ ]:
# ============================================================
# Cell 7: Security Tool -- whois_lookup
# ============================================================
# Simulates a WHOIS query for domain registration data.
# ============================================================

def whois_lookup(domain: str) -> dict:
    """
    Look up WHOIS registration data for a domain.
    Returns registrar, dates, nameservers, and registrant info.
    """
    known_domains = {
        "suspicious-domain.xyz": {
            "domain": "suspicious-domain.xyz",
            "registrar": "NameSilo LLC",
            "creation_date": "2026-01-15",
            "expiration_date": "2027-01-15",
            "last_updated": "2026-01-15",
            "domain_age_days": 46,
            "registrant": {
                "name": "REDACTED FOR PRIVACY",
                "organization": "Privacy service provided by Withheld for Privacy ehf",
                "country": "IS",
                "email": "redacted@witheldforprivacy.com"
            },
            "nameservers": [
                "ns1.shady-dns-provider.ru",
                "ns2.shady-dns-provider.ru"
            ],
            "status": ["clientTransferProhibited"],
            "dnssec": "unsigned",
            "resolved_ips": ["198.51.100.23"],
            "red_flags": [
                "Domain is only 46 days old",
                "Privacy-protected registration",
                "Nameservers in different country than hosting",
                ".xyz TLD commonly abused",
                "DNSSEC not enabled"
            ],
            "confidence": 0.92
        },
        "example-corp.com": {
            "domain": "example-corp.com",
            "registrar": "GoDaddy.com LLC",
            "creation_date": "2015-03-22",
            "expiration_date": "2028-03-22",
            "last_updated": "2025-03-22",
            "domain_age_days": 4000,
            "registrant": {
                "name": "Example Corporation",
                "organization": "Example Corp Inc.",
                "country": "US",
                "email": "domains@example-corp.com"
            },
            "nameservers": [
                "ns1.cloudflare.com",
                "ns2.cloudflare.com"
            ],
            "status": ["clientTransferProhibited", "clientDeleteProhibited"],
            "dnssec": "signedDelegation",
            "resolved_ips": ["104.21.55.123"],
            "red_flags": [],
            "confidence": 0.95
        }
    }

    if domain in known_domains:
        return known_domains[domain]

    # Generic response for unknown domains
    seed = int(hashlib.md5(domain.encode()).hexdigest()[:8], 16)
    rng = random.Random(seed)

    return {
        "domain": domain,
        "registrar": rng.choice(["GoDaddy", "Namecheap", "Cloudflare", "Google Domains"]),
        "creation_date": "2023-06-15",
        "expiration_date": "2026-06-15",
        "domain_age_days": 990,
        "registrant": {
            "name": "REDACTED FOR PRIVACY",
            "organization": "Unknown",
            "country": "US",
            "email": "redacted"
        },
        "nameservers": ["ns1.example.com", "ns2.example.com"],
        "status": ["clientTransferProhibited"],
        "dnssec": "unsigned",
        "resolved_ips": [],
        "red_flags": [],
        "confidence": 0.70
    }

# Test it
print("=== Tool Test: whois_lookup ===")
result = whois_lookup("suspicious-domain.xyz")
print(json.dumps(result, indent=2))

In [ ]:
# ============================================================
# Cell 8: Security Tool -- threat_intel
# ============================================================
# Takes an IOC (IP, hash, or domain) and returns threat
# intelligence enrichment data.
# ============================================================

def threat_intel(ioc: str) -> dict:
    """
    Enrich an Indicator of Compromise (IOC) with threat intelligence.
    Accepts IPs, file hashes (MD5/SHA256), or domains.
    Returns threat context, attribution, and recommendations.
    """
    # Determine IOC type
    if re.match(r"^\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}$", ioc):
        ioc_type = "ip"
    elif re.match(r"^[a-fA-F0-9]{32}$", ioc) or re.match(r"^[a-fA-F0-9]{64}$", ioc):
        ioc_type = "hash"
    else:
        ioc_type = "domain"

    known_iocs = {
        "45.33.32.156": {
            "ioc": "45.33.32.156",
            "ioc_type": "ip",
            "threat_score": 72,
            "first_seen": "2024-03-15",
            "last_seen": "2026-02-28",
            "tags": ["scanner", "botnet", "IoT-exploitation"],
            "associated_campaigns": ["Mirai-variant-2026"],
            "associated_malware": ["Mirai", "Hajime"],
            "attack_patterns": ["T1110 - Brute Force", "T1595 - Active Scanning"],
            "related_iocs": ["45.33.32.157", "45.33.32.158"],
            "attribution": "Unattributed -- likely automated botnet infrastructure",
            "recommendations": [
                "Block at perimeter firewall",
                "Check logs for connections to this IP",
                "Ensure IoT devices are patched and segmented"
            ],
            "sources": ["AlienVault OTX", "VirusTotal", "AbuseIPDB"],
            "confidence": 0.87
        },
        "198.51.100.23": {
            "ioc": "198.51.100.23",
            "ioc_type": "ip",
            "threat_score": 95,
            "first_seen": "2023-08-10",
            "last_seen": "2026-03-01",
            "tags": ["C2", "cobalt-strike", "ransomware-infrastructure"],
            "associated_campaigns": ["BlackCat-affiliate", "Royal-ransomware"],
            "associated_malware": ["Cobalt Strike", "SystemBC", "BlackCat"],
            "attack_patterns": [
                "T1071 - Application Layer Protocol",
                "T1059 - Command and Scripting Interpreter",
                "T1486 - Data Encrypted for Impact"
            ],
            "related_iocs": ["198.51.100.24", "malware-c2.evil.ru",
                             "a1b2c3d4e5f6a1b2c3d4e5f6a1b2c3d4"],
            "attribution": "Suspected Eastern European ransomware affiliate group",
            "recommendations": [
                "IMMEDIATE: Block at all perimeter controls",
                "IMMEDIATE: Search all logs for connections to this IP",
                "Hunt for Cobalt Strike beacons on endpoints",
                "Check for lateral movement indicators",
                "Engage incident response if any connections found"
            ],
            "sources": ["CISA", "Mandiant", "CrowdStrike", "VirusTotal"],
            "confidence": 0.96
        },
        "suspicious-domain.xyz": {
            "ioc": "suspicious-domain.xyz",
            "ioc_type": "domain",
            "threat_score": 91,
            "first_seen": "2026-01-20",
            "last_seen": "2026-03-01",
            "tags": ["C2", "phishing", "newly-registered"],
            "associated_campaigns": ["BlackCat-affiliate"],
            "associated_malware": ["Cobalt Strike", "SystemBC"],
            "attack_patterns": [
                "T1566 - Phishing",
                "T1071 - Application Layer Protocol"
            ],
            "related_iocs": ["198.51.100.23", "198.51.100.24"],
            "attribution": "Suspected Eastern European ransomware affiliate group",
            "recommendations": [
                "Block domain at DNS and proxy",
                "Search email logs for this domain",
                "Check for any user visits to this domain",
                "If visited, initiate incident response"
            ],
            "sources": ["AlienVault OTX", "URLhaus", "PhishTank"],
            "confidence": 0.93
        }
    }

    if ioc in known_iocs:
        return known_iocs[ioc]

    # Generate plausible data for unknown IOCs
    seed = int(hashlib.md5(ioc.encode()).hexdigest()[:8], 16)
    rng = random.Random(seed)
    score = rng.randint(10, 50)

    return {
        "ioc": ioc,
        "ioc_type": ioc_type,
        "threat_score": score,
        "first_seen": "2025-06-01",
        "last_seen": "2026-02-15",
        "tags": [] if score < 30 else ["suspicious"],
        "associated_campaigns": [],
        "associated_malware": [],
        "attack_patterns": [],
        "related_iocs": [],
        "attribution": "No attribution available",
        "recommendations": ["Continue monitoring", "No immediate action required"],
        "sources": ["VirusTotal"],
        "confidence": round(rng.uniform(0.5, 0.75), 2)
    }

# Test it
print("=== Tool Test: threat_intel ===")
result = threat_intel("198.51.100.23")
print(json.dumps(result, indent=2))

---
## Part 2: The Tool Registry
---

The **tool registry** is how we tell the LLM what tools exist. This is the bridge  
between our Python functions and the model's understanding. Each tool gets a  
structured description that the model uses to decide what to call and how.

This is the same concept as:
- **OpenAI function calling** schemas
- **Anthropic tool use** definitions
- **LangChain tool** decorators

We build it from scratch so you see exactly what is happening.

In [ ]:
# ============================================================
# Cell 9: Tool Registry
# ============================================================
# Maps tool names to their functions and describes each tool
# in a format the LLM can understand.
# ============================================================

TOOL_REGISTRY = {
    "ip_reputation": {
        "function": ip_reputation,
        "description": "Look up reputation and geolocation data for an IP address. "
                       "Returns risk score, country, ASN, known malware associations, "
                       "and blocklist status.",
        "parameters": {
            "ip": {
                "type": "string",
                "description": "The IP address to look up (e.g., '45.33.32.156')"
            }
        },
        "returns": "JSON with ip, country, asn, org, risk_score, risk_level, "
                   "malware_families, blocklists, and confidence."
    },
    "cve_lookup": {
        "function": cve_lookup,
        "description": "Look up a CVE vulnerability by its ID. Returns description, "
                       "CVSS score, affected products, exploit availability, and patch status.",
        "parameters": {
            "cve_id": {
                "type": "string",
                "description": "The CVE identifier (e.g., 'CVE-2024-3400')"
            }
        },
        "returns": "JSON with cve_id, description, cvss_score, cvss_severity, "
                   "affected_products, exploit_available, patch_available, and confidence."
    },
    "log_analyzer": {
        "function": log_analyzer,
        "description": "Analyze raw log entries for security anomalies. Detects brute force, "
                       "SQL injection, XSS, path traversal, command injection, and more.",
        "parameters": {
            "log_entries": {
                "type": "string",
                "description": "Raw log entries as a single string, one log line per line."
            }
        },
        "returns": "JSON with total_lines_analyzed, suspicious_ips, summary of findings, "
                   "and detailed findings list with severity levels."
    },
    "port_scanner": {
        "function": port_scanner,
        "description": "Scan a target host or domain for open ports. Returns open ports "
                       "with service identification and version detection.",
        "parameters": {
            "target": {
                "type": "string",
                "description": "Target IP address or domain name to scan."
            }
        },
        "returns": "JSON with target, open_ports list (port, service, version), "
                   "os_guess, and host_status."
    },
    "whois_lookup": {
        "function": whois_lookup,
        "description": "Look up WHOIS registration data for a domain. Returns registrar, "
                       "creation date, registrant info, nameservers, and red flags.",
        "parameters": {
            "domain": {
                "type": "string",
                "description": "Domain name to look up (e.g., 'suspicious-domain.xyz')"
            }
        },
        "returns": "JSON with domain, registrar, creation_date, registrant info, "
                   "nameservers, red_flags list, and confidence."
    },
    "threat_intel": {
        "function": threat_intel,
        "description": "Enrich an Indicator of Compromise (IOC) with threat intelligence. "
                       "Accepts IP addresses, file hashes (MD5/SHA256), or domain names. "
                       "Returns threat context, attribution, and recommendations.",
        "parameters": {
            "ioc": {
                "type": "string",
                "description": "The IOC to look up: IP address, file hash, or domain name."
            }
        },
        "returns": "JSON with threat_score, tags, associated_campaigns, malware, "
                   "MITRE ATT&CK patterns, attribution, and recommendations."
    }
}

def get_tool_descriptions() -> str:
    """Format tool descriptions for the system prompt."""
    lines = []
    for name, info in TOOL_REGISTRY.items():
        params_str = ", ".join(
            f'"{k}": <{v["type"]}> -- {v["description"]}'
            for k, v in info["parameters"].items()
        )
        lines.append(
            f"Tool: {name}\n"
            f"  Description: {info['description']}\n"
            f"  Parameters: {{{params_str}}}\n"
            f"  Returns: {info['returns']}\n"
        )
    return "\n".join(lines)

def execute_tool(tool_name: str, args: dict) -> dict:
    """Execute a tool by name with given arguments. Returns result or error."""
    if tool_name not in TOOL_REGISTRY:
        return {"error": f"Unknown tool: {tool_name}",
                "available_tools": list(TOOL_REGISTRY.keys())}
    try:
        func = TOOL_REGISTRY[tool_name]["function"]
        result = func(**args)
        return result
    except Exception as e:
        return {"error": f"Tool execution failed: {str(e)}",
                "tool": tool_name, "args": args}

# Display the registry
print("=" * 60)
print("SECURITY TOOL REGISTRY")
print("=" * 60)
print(f"Total tools available: {len(TOOL_REGISTRY)}")
print()
for name in TOOL_REGISTRY:
    desc = TOOL_REGISTRY[name]['description'][:80]
    print(f"  [{name}] {desc}...")
print()
print("Tool registry ready.")

---
## Part 3: Building the Agent Loop
---

Now we build the core agent. This is the most important part of the lab.

**The challenge:** Small models (3B parameters) are not as reliable at following  
structured output formats as GPT-4 or Claude. We handle this with:

1. **Clear XML-based tool calling format** -- easier for small models to parse  
2. **Explicit few-shot examples** in the system prompt  
3. **Robust parsing** with regex fallbacks  
4. **Retry logic** -- if the model gives a bad format, we re-prompt  
5. **Maximum iterations** -- prevent infinite loops  

### Tool Call Format

We instruct the model to output tool calls in this format:

```
<tool_call>{"tool": "tool_name", "args": {"param": "value"}}</tool_call>
```

When finished, the model outputs:
```
<final_answer>The complete analysis goes here.</final_answer>
```

This XML-tag approach works much better with small models than pure JSON  
function calling because the tags provide clear delimiters.

In [ ]:
# ============================================================
# Cell 10: System Prompt and Generation Helper
# ============================================================
# The system prompt teaches the model HOW to be an agent.
# This is the most critical piece -- it must be clear enough
# for a 3B model to follow reliably.
# ============================================================

SYSTEM_PROMPT = f"""You are a Security Investigation Agent. You help security analysts investigate threats by using specialized security tools.

## Available Tools

{get_tool_descriptions()}

## How to Use Tools

When you need information, call a tool using this EXACT format:

<tool_call>{{"tool": "tool_name", "args": {{"param_name": "value"}}}}</tool_call>

IMPORTANT RULES:
1. Call ONE tool at a time.
2. After calling a tool, STOP and wait for the result.
3. After receiving a tool result, reason about what you learned and decide if you need more information.
4. When you have enough information to answer, provide your final analysis using:

<final_answer>
Your complete analysis here.
</final_answer>

## Examples

User: What can you tell me about IP 8.8.8.8?
Assistant: I will look up the reputation data for this IP address.

<tool_call>{{"tool": "ip_reputation", "args": {{"ip": "8.8.8.8"}}}}</tool_call>

[After receiving result]
The IP reputation data shows this is a Google DNS server. Let me also check threat intelligence.

<tool_call>{{"tool": "threat_intel", "args": {{"ioc": "8.8.8.8"}}}}</tool_call>

[After receiving result]
<final_answer>
IP 8.8.8.8 is Google's public DNS resolver. Risk score is very low. No malicious activity associated.
</final_answer>

## Investigation Approach

Think like a security analyst:
1. Start with the most relevant tool for the query
2. Use results to guide follow-up investigations
3. Cross-reference findings across multiple tools
4. Synthesize all findings into a clear, actionable report
5. Include risk assessment and recommendations

Always explain your reasoning before each tool call."""


def generate_response(messages, max_new_tokens=512):
    """
    Generate a response from the model given a list of messages.
    Uses the chat template for proper formatting.
    """
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.3,       # Low temperature for reliability
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Decode only the new tokens (skip the input)
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True)
    return response.strip()


print("System prompt length:", len(SYSTEM_PROMPT), "characters")
print("Generation helper ready.")
print()
print("First 500 chars of system prompt:")
print("-" * 40)
print(SYSTEM_PROMPT[:500])
print("...")

In [ ]:
# ============================================================
# Cell 11: Tool Call Parser
# ============================================================
# Robust parsing of tool calls from model output.
# Handles malformed JSON, missing tags, and other issues
# that small models commonly produce.
# ============================================================

def parse_tool_call(response: str) -> dict | None:
    """
    Parse a tool call from the model's response.

    Returns a dict with 'tool' and 'args' keys, or None if no
    valid tool call is found.

    Handles multiple common failure modes:
    1. Proper XML tags with valid JSON
    2. XML tags with slightly malformed JSON (single quotes, etc.)
    3. JSON-like patterns without XML tags
    4. Tool name mentioned in text without proper format
    """

    # Strategy 1: Proper <tool_call> tags
    match = re.search(r"<tool_call>(.*?)</tool_call>", response, re.DOTALL)
    if match:
        raw = match.group(1).strip()
        try:
            parsed = json.loads(raw)
            if "tool" in parsed and "args" in parsed:
                return parsed
        except json.JSONDecodeError:
            pass

        # Try fixing common JSON issues
        try:
            # Replace single quotes with double quotes
            fixed = raw.replace("'", '"')
            parsed = json.loads(fixed)
            if "tool" in parsed and "args" in parsed:
                return parsed
        except json.JSONDecodeError:
            pass

    # Strategy 2: Look for JSON-like tool call pattern anywhere in response
    match = re.search(
        r'\{\s*"tool"\s*:\s*"(\w+)"\s*,\s*"args"\s*:\s*(\{[^}]+\})\s*\}',
        response, re.DOTALL
    )
    if match:
        tool_name = match.group(1)
        try:
            args = json.loads(match.group(2))
            return {"tool": tool_name, "args": args}
        except json.JSONDecodeError:
            pass

    # Strategy 3: Look for tool name + argument pattern (very lenient)
    for tool_name in TOOL_REGISTRY:
        if tool_name in response:
            # Try to extract the first quoted string as the argument
            params = TOOL_REGISTRY[tool_name]["parameters"]
            param_name = list(params.keys())[0]

            # Look for quoted values near the tool name
            pattern = tool_name + r'.*?["\']([^"\']+ )["\']'
            arg_match = re.search(pattern, response, re.DOTALL)
            if arg_match:
                return {
                    "tool": tool_name,
                    "args": {param_name: arg_match.group(1)}
                }

    return None


def parse_final_answer(response: str) -> str | None:
    """
    Extract the final answer from model output.
    Returns the answer text or None if not found.
    """
    match = re.search(r"<final_answer>(.*?)</final_answer>", response, re.DOTALL)
    if match:
        return match.group(1).strip()

    # If no explicit tag but response has substantial text and no tool call,
    # treat the whole response as a final answer (fallback)
    if not parse_tool_call(response) and len(response) > 100:
        return response.strip()

    return None


# Test the parser
print("=== Parser Tests ===")

test_cases = [
    # Good format
    'I will look up this IP.\n<tool_call>{"tool": "ip_reputation", "args": {"ip": "1.2.3.4"}}</tool_call>',
    # Single quotes (common model error)
    "<tool_call>{'tool': 'cve_lookup', 'args': {'cve_id': 'CVE-2024-3400'}}</tool_call>",
    # No XML tags but valid JSON pattern
    'Let me check that. {"tool": "threat_intel", "args": {"ioc": "evil.com"}}',
    # Final answer
    '<final_answer>The IP is malicious. Block it immediately.</final_answer>',
]

for i, test in enumerate(test_cases):
    tc = parse_tool_call(test)
    fa = parse_final_answer(test)
    print(f"\nTest {i+1}:")
    print(f"  Input: {test[:80]}...")
    print(f"  Tool Call: {tc}")
    print(f"  Final Answer: {fa[:50] if fa else None}")

print("\nParser ready.")

In [ ]:
# ============================================================
# Cell 12: The Agent Loop
# ============================================================
# This is the CORE of the agent. It implements the full
# observe-think-act loop with tool execution and multi-step
# reasoning.
# ============================================================

def run_agent(user_query: str, max_iterations: int = 6, verbose: bool = True) -> str:
    """
    Run the security agent on a user query.

    The agent loop:
    1. Send query to LLM with system prompt and tool descriptions
    2. Parse the response for tool calls or final answer
    3. If tool call: execute the tool, feed result back, repeat
    4. If final answer: return it
    5. If max iterations reached: force a final summary

    Args:
        user_query: The security question to investigate
        max_iterations: Maximum tool-calling rounds (prevents infinite loops)
        verbose: Print detailed agent reasoning

    Returns:
        The agent's final analysis as a string
    """
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_query}
    ]

    if verbose:
        print("=" * 60)
        print("SECURITY AGENT -- INVESTIGATION STARTED")
        print("=" * 60)
        print(f"Query: {user_query}")
        print("=" * 60)

    tools_called = []

    for iteration in range(max_iterations):
        if verbose:
            print(f"\n--- Iteration {iteration + 1}/{max_iterations} ---")
            print("Generating agent response...")

        # Generate response from the model
        response = generate_response(messages, max_new_tokens=512)

        if verbose:
            print(f"\nAgent output:\n{response}")

        # Check for final answer
        final = parse_final_answer(response)
        tool_call = parse_tool_call(response)

        if final and not tool_call:
            if verbose:
                print("\n" + "=" * 60)
                print("INVESTIGATION COMPLETE")
                print("=" * 60)
                print(f"Tools used: {tools_called}")
            return final

        # Check for tool call
        if tool_call:
            tool_name = tool_call["tool"]
            tool_args = tool_call["args"]

            if verbose:
                print(f"\n>> TOOL CALL: {tool_name}({json.dumps(tool_args)})")

            # Execute the tool
            result = execute_tool(tool_name, tool_args)
            tools_called.append(tool_name)

            if verbose:
                result_preview = json.dumps(result, indent=2, default=str)
                if len(result_preview) > 500:
                    result_preview = result_preview[:500] + "\n  ... (truncated)"
                print(f">> RESULT:\n{result_preview}")

            # Add assistant response and tool result to conversation
            messages.append({"role": "assistant", "content": response})
            messages.append({
                "role": "user",
                "content": f"Tool result for {tool_name}:\n{json.dumps(result, indent=2, default=str)}\n\n"
                           f"Based on this result, continue your investigation. "
                           f"Call another tool if needed, or provide your <final_answer> if you have enough information."
            })
            continue

        # No tool call and no final answer -- the model might be confused.
        # Add a nudge to guide it.
        if verbose:
            print("\n[No tool call or final answer detected. Nudging the agent...]")

        messages.append({"role": "assistant", "content": response})
        messages.append({
            "role": "user",
            "content": "Please either call a tool using <tool_call>{...}</tool_call> format, "
                       "or provide your final analysis using <final_answer>...</final_answer> format."
        })

    # Max iterations reached -- force a final summary
    if verbose:
        print(f"\n[Max iterations ({max_iterations}) reached. Forcing final summary...]")

    messages.append({
        "role": "user",
        "content": "Maximum investigation steps reached. Please provide your "
                   "<final_answer> now with everything you have learned so far."
    })
    response = generate_response(messages, max_new_tokens=512)
    final = parse_final_answer(response)

    if verbose:
        print("\n" + "=" * 60)
        print("INVESTIGATION COMPLETE (max iterations)")
        print("=" * 60)
        print(f"Tools used: {tools_called}")

    return final if final else response


print("Agent loop ready.")
print("Use: run_agent('your security question here')")

---
## Part 4: Demo Investigations
---

Now let us put the agent to work. We will run four increasingly complex  
investigations to see how the agent reasons, selects tools, and chains results.

**Watch for:**
- How the agent decides which tool to call first
- How it uses results from one tool to inform the next call
- How it synthesizes multiple data sources into a coherent report
- Where it struggles (this is a 3B model -- it will not be perfect!)

In [ ]:
# ============================================================
# Cell 13: Demo 1 -- Simple IP Investigation
# ============================================================
# A straightforward query that should trigger ip_reputation
# and possibly threat_intel.
# ============================================================

print("\n" + "#" * 60)
print("# DEMO 1: Simple IP Investigation")
print("#" * 60)

demo1_result = run_agent(
    "What can you tell me about IP 45.33.32.156? Is it malicious?",
    max_iterations=4,
    verbose=True
)

print("\n" + "=" * 60)
print("FINAL REPORT:")
print("=" * 60)
print(demo1_result)

### Demo 1 Discussion

**What happened:**
- The agent should have called `ip_reputation` first (most relevant tool)
- It may have followed up with `threat_intel` for enrichment
- It synthesized the results into a risk assessment

**What to notice:**
- Did the agent explain its reasoning before each tool call?
- Did it correctly identify the IP as high-risk?
- Did it provide actionable recommendations?

**If the agent failed:** Small models sometimes struggle with the format on the  
first try. The retry logic should handle most cases. If it still fails, the  
nudging mechanism forces it back on track.

In [ ]:
# ============================================================
# Cell 14: Demo 2 -- CVE Risk Assessment
# ============================================================
# A multi-step query: look up the CVE, assess risk for a
# specific product, check exploit availability.
# ============================================================

print("\n" + "#" * 60)
print("# DEMO 2: CVE Risk Assessment")
print("#" * 60)

demo2_result = run_agent(
    "I found CVE-2024-3400 in our vulnerability scan results. "
    "We have Palo Alto GlobalProtect firewalls running PAN-OS 11.0.3. "
    "Assess the risk and tell me what we need to do.",
    max_iterations=4,
    verbose=True
)

print("\n" + "=" * 60)
print("FINAL REPORT:")
print("=" * 60)
print(demo2_result)

### Demo 2 Discussion

**What happened:**
- The agent should have called `cve_lookup` to get CVE details
- It should have assessed that PAN-OS 11.0.3 is in the affected range (< 11.0.4-h1)
- It should have noted the CVSS 10.0 score, active exploitation, and exploit availability
- A good agent response would include urgency and specific remediation steps

**Key learning:** The agent can combine structured data (from the tool) with its  
pre-trained knowledge (about firewall patching procedures) to give a complete answer.

In [ ]:
# ============================================================
# Cell 15: Demo 3 -- Incident Triage (Log Analysis)
# ============================================================
# Feed suspicious logs to the agent. It should analyze them
# and then follow up by investigating the suspicious IPs.
# ============================================================

print("\n" + "#" * 60)
print("# DEMO 3: Incident Triage")
print("#" * 60)

suspicious_logs = """2026-03-01 08:14:22 sshd[12001]: Failed password for root from 198.51.100.23 port 4822 ssh2
2026-03-01 08:14:23 sshd[12002]: Failed password for admin from 198.51.100.23 port 4823 ssh2
2026-03-01 08:14:24 sshd[12003]: Failed password for root from 198.51.100.23 port 4824 ssh2
2026-03-01 08:14:25 sshd[12004]: Failed password for root from 198.51.100.23 port 4825 ssh2
2026-03-01 08:14:26 sshd[12005]: Failed password for admin from 198.51.100.23 port 4826 ssh2
2026-03-01 08:14:27 sshd[12006]: Failed password for root from 198.51.100.23 port 4827 ssh2
2026-03-01 08:15:01 httpd[5501]: 203.0.113.50 - - "GET /admin/../../../etc/passwd HTTP/1.1" 403
2026-03-01 08:15:05 httpd[5502]: 203.0.113.50 - - "POST /login HTTP/1.1" 200 -- 'admin' OR '1'='1'
2026-03-01 08:15:10 httpd[5503]: 203.0.113.50 - - "GET /admin/cmd.exe?/c+whoami HTTP/1.1" 404
2026-03-01 08:16:00 kernel: [UFW BLOCK] SRC=198.51.100.23 DST=10.0.1.5 PROTO=TCP DPT=4444"""

demo3_result = run_agent(
    f"Analyze these logs from our production server and identify potential threats. "
    f"Investigate any suspicious IPs you find.\n\nLogs:\n{suspicious_logs}",
    max_iterations=6,
    verbose=True
)

print("\n" + "=" * 60)
print("FINAL REPORT:")
print("=" * 60)
print(demo3_result)

### Demo 3 Discussion

**Expected agent behavior (multi-step):**
1. Call `log_analyzer` on the raw logs to identify anomalies
2. See suspicious IPs (198.51.100.23 and 203.0.113.50) in the results
3. Call `ip_reputation` or `threat_intel` on 198.51.100.23 (brute force source)
4. Possibly call `ip_reputation` on 203.0.113.50 (web attack source)
5. Synthesize everything into an incident report

**This demonstrates chaining:** The agent uses output from one tool (suspicious IPs  
from log analysis) as input to another tool (IP reputation lookup). This is the  
core value of agentic AI -- it can follow investigation threads automatically.

**Scoring rubric for a good response:**
- Identified brute force campaign from 198.51.100.23
- Identified web attacks (path traversal, SQLi, command injection) from 203.0.113.50
- Noted the blocked connection attempt to port 4444 (possible C2)
- Provided severity assessment
- Gave specific remediation recommendations

In [ ]:
# ============================================================
# Cell 16: Demo 4 -- Full Investigation Chain
# ============================================================
# A complex scenario that should trigger 3-4 different tools
# in sequence: whois -> ip_reputation -> threat_intel -> report
# ============================================================

print("\n" + "#" * 60)
print("# DEMO 4: Full Investigation Chain")
print("#" * 60)

demo4_result = run_agent(
    "Our IDS flagged outbound HTTPS traffic to suspicious-domain.xyz from "
    "multiple internal workstations. This domain was not in our allowlist. "
    "Conduct a full investigation of this domain and assess the threat level.",
    max_iterations=6,
    verbose=True
)

print("\n" + "=" * 60)
print("FINAL REPORT:")
print("=" * 60)
print(demo4_result)

### Demo 4 Discussion

**Expected investigation chain:**
1. `whois_lookup("suspicious-domain.xyz")` -- Check domain registration
   - Discovers: domain is only 46 days old, privacy-protected, sketchy nameservers
2. `ip_reputation("198.51.100.23")` -- Check the resolved IP
   - Discovers: CRITICAL risk, C2 server, associated with ransomware
3. `threat_intel("suspicious-domain.xyz")` or `threat_intel("198.51.100.23")` -- Get threat context
   - Discovers: BlackCat affiliate, Cobalt Strike, ransomware infrastructure
4. Possibly `port_scanner("suspicious-domain.xyz")` -- Check what is running
   - Discovers: port 4444 and 8443 (Cobalt Strike beacon!)

**This is the power of agentic AI:** A human analyst would follow the same steps,  
but the agent does it in seconds and provides a structured report.

**The critical question:** Should this agent be allowed to automatically block this  
domain? We explore that next.

---
## Part 5: Agent Improvements
---

Our basic agent works, but production agents need several improvements.  
Let us implement the most important ones.

In [ ]:
# ============================================================
# Cell 17: Improved Agent with Memory and Error Handling
# ============================================================
# Adds conversation history (memory), confidence tracking,
# and robust error handling for tool failures.
# ============================================================

class SecurityAgent:
    """
    An improved security investigation agent with:
    - Conversation memory (remembers past investigations)
    - Confidence tracking across tool results
    - Error handling and graceful degradation
    - Investigation logging
    """

    def __init__(self, model, tokenizer, tool_registry, verbose=True):
        self.model = model
        self.tokenizer = tokenizer
        self.tools = tool_registry
        self.verbose = verbose
        self.conversation_history = []  # Memory across investigations
        self.investigation_log = []     # Detailed log of current investigation

    def _generate(self, messages, max_new_tokens=512):
        """Generate a response from the model."""
        text = self.tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = self.tokenizer(text, return_tensors="pt").to(self.model.device)

        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=0.3,
                top_p=0.9,
                do_sample=True,
                repetition_penalty=1.1,
                pad_token_id=self.tokenizer.eos_token_id,
            )

        new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
        return self.tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    def _execute_tool(self, tool_name, args):
        """Execute a tool with error handling."""
        if tool_name not in self.tools:
            return {
                "error": f"Unknown tool '{tool_name}'",
                "available_tools": list(self.tools.keys()),
                "confidence": 0.0
            }

        try:
            func = self.tools[tool_name]["function"]
            result = func(**args)

            # Log the tool execution
            self.investigation_log.append({
                "timestamp": datetime.now().isoformat(),
                "action": "tool_call",
                "tool": tool_name,
                "args": args,
                "success": True,
                "confidence": result.get("confidence", "N/A")
            })
            return result

        except TypeError as e:
            error_result = {
                "error": f"Invalid arguments for {tool_name}: {str(e)}",
                "expected_params": list(self.tools[tool_name]["parameters"].keys()),
                "received_args": args,
                "confidence": 0.0
            }
            self.investigation_log.append({
                "timestamp": datetime.now().isoformat(),
                "action": "tool_error",
                "tool": tool_name,
                "error": str(e)
            })
            return error_result

        except Exception as e:
            error_result = {
                "error": f"Tool execution failed: {str(e)}",
                "confidence": 0.0
            }
            self.investigation_log.append({
                "timestamp": datetime.now().isoformat(),
                "action": "tool_error",
                "tool": tool_name,
                "error": str(e)
            })
            return error_result

    def investigate(self, query, max_iterations=6):
        """
        Run a full investigation with memory and logging.
        """
        self.investigation_log = []
        tools_used = []
        confidence_scores = []

        # Build context with memory of past investigations
        memory_context = ""
        if self.conversation_history:
            recent = self.conversation_history[-3:]  # Last 3 investigations
            memory_context = "\n\nPrevious investigations for context:\n"
            for past in recent:
                memory_context += f"- Query: {past['query'][:100]}\n"
                memory_context += f"  Key finding: {past['summary'][:150]}\n"

        messages = [
            {"role": "system", "content": SYSTEM_PROMPT + memory_context},
            {"role": "user", "content": query}
        ]

        if self.verbose:
            print("=" * 60)
            print("SECURITY AGENT v2 -- INVESTIGATION STARTED")
            print("=" * 60)
            print(f"Query: {query}")
            if memory_context:
                print(f"Memory: {len(self.conversation_history)} past investigations loaded")
            print("=" * 60)

        for iteration in range(max_iterations):
            if self.verbose:
                print(f"\n--- Iteration {iteration + 1}/{max_iterations} ---")

            response = self._generate(messages)

            if self.verbose:
                print(f"\nAgent:\n{response}")

            # Check for final answer
            final = parse_final_answer(response)
            tool_call = parse_tool_call(response)

            if final and not tool_call:
                # Save to memory
                self.conversation_history.append({
                    "query": query,
                    "summary": final[:200],
                    "tools_used": tools_used,
                    "avg_confidence": (
                        sum(confidence_scores) / len(confidence_scores)
                        if confidence_scores else 0
                    ),
                    "timestamp": datetime.now().isoformat()
                })

                if self.verbose:
                    print("\n" + "=" * 60)
                    print("INVESTIGATION COMPLETE")
                    print(f"Tools used: {tools_used}")
                    if confidence_scores:
                        print(f"Average confidence: {sum(confidence_scores)/len(confidence_scores):.2f}")
                    print("=" * 60)

                return final

            if tool_call:
                tool_name = tool_call["tool"]
                tool_args = tool_call["args"]

                if self.verbose:
                    print(f"\n>> TOOL: {tool_name}({json.dumps(tool_args)})")

                result = self._execute_tool(tool_name, tool_args)
                tools_used.append(tool_name)

                # Track confidence
                if "confidence" in result and isinstance(result["confidence"], (int, float)):
                    confidence_scores.append(result["confidence"])

                if self.verbose:
                    preview = json.dumps(result, indent=2, default=str)
                    if len(preview) > 400:
                        preview = preview[:400] + "\n  ..."
                    print(f">> RESULT:\n{preview}")

                messages.append({"role": "assistant", "content": response})
                messages.append({
                    "role": "user",
                    "content": f"Tool result for {tool_name}:\n"
                               f"{json.dumps(result, indent=2, default=str)}\n\n"
                               f"Continue investigating. Call another tool or provide your <final_answer>."
                })
                continue

            # Nudge
            messages.append({"role": "assistant", "content": response})
            messages.append({
                "role": "user",
                "content": "Use <tool_call>...</tool_call> to call a tool or "
                           "<final_answer>...</final_answer> to give your analysis."
            })

        # Force final
        messages.append({
            "role": "user",
            "content": "Provide your <final_answer> now with all findings so far."
        })
        response = self._generate(messages)
        final = parse_final_answer(response) or response

        self.conversation_history.append({
            "query": query,
            "summary": final[:200],
            "tools_used": tools_used,
            "avg_confidence": (
                sum(confidence_scores) / len(confidence_scores)
                if confidence_scores else 0
            ),
            "timestamp": datetime.now().isoformat()
        })

        return final

    def get_investigation_log(self):
        """Return the detailed log of the most recent investigation."""
        return self.investigation_log

    def get_memory_summary(self):
        """Return a summary of all past investigations."""
        return [
            {
                "query": inv["query"][:80],
                "tools": inv["tools_used"],
                "confidence": round(inv["avg_confidence"], 2),
                "time": inv["timestamp"]
            }
            for inv in self.conversation_history
        ]


# Create the improved agent
agent = SecurityAgent(model, tokenizer, TOOL_REGISTRY, verbose=True)
print("SecurityAgent v2 initialized.")
print("Use: agent.investigate('your query')")

In [ ]:
# ============================================================
# Cell 18: Test the Improved Agent
# ============================================================
# Run two investigations to demonstrate memory.
# ============================================================

print("\n" + "#" * 60)
print("# Testing Agent v2 with Memory")
print("#" * 60)

# Investigation 1
result1 = agent.investigate(
    "Check the reputation of IP 198.51.100.23",
    max_iterations=3
)
print("\nFINAL REPORT 1:")
print(result1)

print("\n" + "-" * 40)

# Investigation 2 -- agent now has memory of investigation 1
result2 = agent.investigate(
    "We also found traffic to suspicious-domain.xyz from the same hosts. Investigate this domain.",
    max_iterations=4
)
print("\nFINAL REPORT 2:")
print(result2)

# Show memory
print("\n" + "=" * 60)
print("AGENT MEMORY:")
print("=" * 60)
for entry in agent.get_memory_summary():
    print(json.dumps(entry, indent=2))

---
## Part 6: Excessive Agency -- The Critical Risk
---

### The Question That Matters Most

Our agent can **read** data: look up IPs, search CVEs, analyze logs. These are  
**safe, read-only operations**. But what if we gave it tools that **take action**?

### Imagine these tools existed:

```python
def block_ip(ip, firewall="perimeter"):
    """Block an IP at the firewall. IRREVERSIBLE without manual review."""
    ...

def quarantine_host(hostname):
    """Isolate a host from the network. Disrupts user productivity."""
    ...

def disable_user(username):
    """Disable an Active Directory account. User cannot work."""
    ...

def deploy_patch(cve_id, target_hosts):
    """Push emergency patch to hosts. May cause service restart."""
    ...
```

### What Could Go Wrong?

| Scenario | What Happens | Impact |
|----------|-------------|--------|
| False positive IP block | Agent blocks legitimate CDN IP | Website goes down for all users |
| Quarantine mistake | Agent isolates CEO's laptop | Executive cannot present to board |
| Account disable | Agent disables service account | Production services crash |
| Aggressive patching | Agent pushes patch during peak hours | Service disruption for thousands |

### OWASP LLM08: Excessive Agency

From the OWASP Top 10 for LLM Applications:

> *"An LLM-based system is often granted a degree of agency by its developer --  
> the ability to call functions or interface with other systems via extensions  
> to generate responses. The decision over which extensions to invoke may also  
> be delegated to an LLM 'agent' to dynamically determine based on input prompt  
> or LLM output.*  
>
> *Excessive Agency is the vulnerability that enables damaging actions to be  
> performed in response to unexpected, ambiguous, or manipulated outputs from  
> an LLM, regardless of what is causing the LLM to malfunction."*

### The Rule: Read vs. Write

```
+------------------------------------------+
|          AGENT TOOL CATEGORIES            |
+------------------------------------------+
|                                          |
|   READ (Safe for automation):            |
|   - ip_reputation    - cve_lookup        |
|   - log_analyzer     - port_scanner      |
|   - whois_lookup     - threat_intel      |
|                                          |
|   WRITE (Require human approval):        |
|   - block_ip         - quarantine_host   |
|   - disable_user     - deploy_patch      |
|   - send_alert       - modify_firewall   |
|                                          |
+------------------------------------------+
```

### Mitigation: Human-in-the-Loop

For any tool that **changes state**, the agent should:
1. **Recommend** the action ("I recommend blocking 198.51.100.23")
2. **Explain** why ("This IP has a 95/100 risk score, associated with ransomware")
3. **Wait for human approval** before executing
4. **Log the decision** (who approved, when, why)

**Think about it:** Would you trust a 3B parameter model to decide whether to  
block an IP on your production firewall? What about GPT-4? What about a model  
that has been prompt-injected?

This is why **excessive agency** is in the OWASP Top 10.

In [ ]:
# ============================================================
# Cell 19: Human-in-the-Loop Demonstration
# ============================================================
# Show what a responsible "write" action looks like:
# agent recommends, human confirms.
# ============================================================

def block_ip_with_approval(ip: str, reason: str, auto_approve: bool = False) -> dict:
    """
    Simulates a firewall block with human-in-the-loop.
    In production, this would pause and wait for analyst confirmation.
    """
    print("\n" + "!" * 60)
    print("!! HUMAN APPROVAL REQUIRED !!")
    print("!" * 60)
    print(f"\n  Action: BLOCK IP at perimeter firewall")
    print(f"  Target: {ip}")
    print(f"  Reason: {reason}")
    print(f"  Reversible: Yes (manual unblock required)")
    print(f"  Impact: All traffic from {ip} will be dropped")
    print()

    if auto_approve:
        print("  [AUTO-APPROVED for demonstration]")
        approved = True
    else:
        # In a real system, this would be a web UI, Slack message, etc.
        print("  In production: Analyst would see this in their SOC dashboard")
        print("  and click [APPROVE] or [REJECT]")
        approved = True  # Simulated approval

    print("!" * 60)

    if approved:
        return {
            "action": "block_ip",
            "status": "EXECUTED",
            "target": ip,
            "reason": reason,
            "approved_by": "analyst@soc.example.com",
            "approved_at": datetime.now().isoformat(),
            "rule_id": f"FW-{random.randint(10000, 99999)}",
            "note": "This was a SIMULATED block for demonstration purposes."
        }
    else:
        return {
            "action": "block_ip",
            "status": "REJECTED",
            "target": ip,
            "rejected_by": "analyst@soc.example.com",
            "reason": "Analyst determined block was not warranted."
        }

# Demonstrate the human-in-the-loop pattern
print("Demonstrating human-in-the-loop for a 'write' action:\n")
result = block_ip_with_approval(
    ip="198.51.100.23",
    reason="CRITICAL risk score (95/100). C2 server associated with BlackCat ransomware. "
           "Active connections observed from internal hosts.",
    auto_approve=True
)
print("\nResult:")
print(json.dumps(result, indent=2))

---
## Part 7: Student Exercises
---

Now it is your turn. Complete the following exercises to deepen your understanding  
of agentic AI and security tool calling.

### Exercise 1: Add a New Security Tool

Create a new tool called `hash_lookup` that:
- Takes a file hash (MD5 or SHA256) as input
- Returns: file name, file type, detection count (AV engines), malware family,  
  first/last seen dates, and sandbox behavior notes
- Add it to the `TOOL_REGISTRY`
- Test it with: `agent.investigate("Check if this hash is malicious: a1b2c3d4e5f6...")`

In [ ]:
# ============================================================
# Exercise 1: Add hash_lookup tool
# ============================================================
# YOUR CODE HERE
#
# def hash_lookup(file_hash: str) -> dict:
#     """Look up a file hash in threat intelligence databases."""
#     # Hint: Create a few known malicious hashes with realistic data
#     # and a fallback for unknown hashes
#     pass
#
# Don't forget to add it to TOOL_REGISTRY!
# ============================================================

print("Exercise 1: Implement hash_lookup and add to TOOL_REGISTRY")
print("Then test with: agent.investigate('Check hash a1b2c3d4...')")

### Exercise 2: Multi-Tool Investigation Scenario

Create an investigation scenario that requires **at least 3 tool calls** to resolve.  
Write the scenario description and then run the agent on it.

Example structure:
- A phishing email was received containing a suspicious link
- The agent needs to: check the domain (whois), check the IP (ip_reputation),  
  and get threat intelligence
- Bonus: include a CVE that relates to the attack vector

In [ ]:
# ============================================================
# Exercise 2: Multi-tool scenario
# ============================================================
# YOUR CODE HERE
#
# scenario = """
# [Write your investigation scenario here. Make it realistic
#  and require multiple tools to fully investigate.]
# """
#
# result = agent.investigate(scenario, max_iterations=6)
# print(result)
# ============================================================

print("Exercise 2: Create a scenario requiring 3+ tool calls.")
print("Run it with agent.investigate() and observe the tool chain.")

### Exercise 3: Add a Confirmation Step for Write Actions

Modify the agent so that any tool with a `requires_approval: True` flag in the  
registry triggers a confirmation step before execution.

Steps:
1. Add `block_ip_with_approval` to the tool registry with `requires_approval: True`
2. Modify `_execute_tool` in `SecurityAgent` to check for the flag
3. If the flag is set, print a warning and simulate waiting for approval
4. Test by asking the agent to block a malicious IP

In [ ]:
# ============================================================
# Exercise 3: Confirmation step for write actions
# ============================================================
# YOUR CODE HERE
#
# Hint: Modify SecurityAgent._execute_tool to check:
#   if self.tools[tool_name].get("requires_approval", False):
#       # Print warning, simulate approval request
#       # Only execute if approved
#
# Then add a "dangerous" tool to the registry:
# TOOL_REGISTRY["block_ip"] = {
#     "function": block_ip_with_approval,
#     "description": "Block an IP at the perimeter firewall.",
#     "parameters": {...},
#     "requires_approval": True,
# }
# ============================================================

print("Exercise 3: Add approval gates for dangerous tools.")
print("This is how production security agents should work.")

---
## Key Takeaways
---

### What You Built
- A **functional AI security agent** that uses tool calling to investigate threats
- Six security tools with realistic simulated data
- A **robust parsing system** that handles messy LLM output
- An **agentic loop** with memory, error handling, and iteration limits

### What You Learned

1. **The Agentic Loop is simple.** At its core: generate -> parse -> execute -> feed back.  
   The complexity is in making it reliable.

2. **Small models can do tool calling,** but they need clear formats (XML tags),  
   few-shot examples, and robust parsing with fallbacks.

3. **Tool selection is reasoning.** The model must understand what each tool does  
   and decide which one(s) to use. This is a form of planning.

4. **Chaining is powerful.** The real value comes when the agent uses output from  
   one tool as input to another, building up a complete picture.

5. **Excessive agency is dangerous.** Read-only tools are safe to automate.  
   Write/action tools need human-in-the-loop gates.

### Connection to Other Labs

| Lab | Connection |
|-----|------------|
| Lab 1 (Prompt Injection) | What if someone injects a prompt that tricks the agent into calling the wrong tool? |
| Lab 3 (Guardrails) | Guardrails can filter dangerous tool call requests before execution |
| Lab 4 (Abliteration) | An abliterated model used as an agent brain has NO safety filters on tool calls |
| Lab 7 (Jailbreaking) | A jailbroken agent could be convinced to call destructive tools |

### The Big Picture

AI agents are the future of security operations. They will:
- Automate tier-1 SOC analysis
- Triage thousands of alerts per day
- Correlate threat intelligence at machine speed

But they must be built with **appropriate constraints**. The tools you grant  
to an agent define its **capability envelope**. A read-only agent is a useful  
assistant. An unconstrained agent with write access is an **insider threat**.

**Build agents that help humans decide, not agents that decide for humans.**

---

*Lab 9 Complete. Derek Banks and Dr. Brian Fehrman, 2026.*